# Car Price Analysis — Machine Learning Assignment

**Objective:** Explore a used-car pricing dataset, build a **Linear Regression** model to predict price, and a **KNN Classifier** to categorise cars as Cheap / Moderate / Expensive.

**Dataset:** `car price.csv` — ~72 000 UK used-car listings with 10 features.

---

## 0. Library Imports

In [1]:
# Install all required libraries (run once, then restart kernel if needed)
!pip install numpy pandas matplotlib seaborn scikit-learn scipy

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings('ignore')

# Consistent plot style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

---
## 1. Exploratory Data Analysis (EDA)

We start by loading the dataset and understanding its size, data types, missing values, and distributions before any transformations.

### 1.1 Load and Inspect the Dataset

In [3]:
df = pd.read_csv('car price.csv')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
df.head(10)

Shape: 72435 rows x 10 columns


,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,Make
0,A1,2017.0,12500.0,Manual,15735.0,Petrol,150.0,55.4,1.4,audi
1,A6,2016.0,16500.0,Automatic,36203.0,Diesel,20.0,64.2,2.0,audi
2,A1,2016.0,11000.0,Manual,29946.0,Petrol,30.0,55.4,1.4,audi
3,A4,2017.0,16800.0,Automatic,25952.0,Diesel,145.0,67.3,2.0,audi
4,A3,2019.0,17300.0,Manual,1998.0,Petrol,145.0,49.6,1.0,audi
5,A1,2016.0,13900.0,Automatic,32260.0,Petrol,30.0,58.9,1.4,audi
6,A6,2016.0,13250.0,Automatic,76788.0,Diesel,30.0,61.4,2.0,audi
7,A4,2016.0,11750.0,NaN,75185.0,Diesel,20.0,70.6,2.0,audi
8,A3,2015.0,10200.0,Manual,46112.0,Petrol,20.0,60.1,1.4,audi
9,A1,2016.0,12000.0,Manual,22451.0,NaN,30.0,55.4,1.4,audi


In [ ]:
df.info()

In [ ]:
df.describe()

### 1.2 Identify Numerical vs Categorical Features

In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()

print(f'Numerical  ({len(numerical_cols)}): {numerical_cols}')
print(f'Categorical ({len(categorical_cols)}): {categorical_cols}')

### 1.3 Missing Values Analysis

Missing values can bias a model or cause it to crash. We count them per column and decide on a handling strategy in the preprocessing section.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
print(f'Total missing values: {df.isnull().sum().sum()}')
missing_df

### 1.4 Price Distribution (Histogram)

Understanding the target variable's shape helps us choose appropriate thresholds for classification later and spot potential outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['price'].dropna(), bins=50, edgecolor='black', color='steelblue', alpha=0.8)
axes[0].set_title('Price Distribution', fontsize=14)
axes[0].set_xlabel('Price')
axes[0].set_ylabel('Frequency')

# Box plot for outlier view
axes[1].boxplot(df['price'].dropna(), vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Price Box Plot', fontsize=14)
axes[1].set_xlabel('Price')

plt.tight_layout()
plt.show()

print(f"Price statistics:\n{df['price'].describe()}")

**Insight:** Price is right-skewed — most cars are priced under 30 000, but there is a long tail of expensive cars. The box plot confirms heavy outliers on the upper end.

### 1.5 Correlation Heatmap

We examine pairwise Pearson correlations among numerical features to understand which features relate most strongly to price and to detect multicollinearity.

In [ ]:
corr = df[numerical_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.4, annot_kws={'size': 9})
plt.title('Pearson Correlation Matrix — Numerical Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Features ranked by absolute correlation with price
price_corr = corr['price'].drop('price').abs().sort_values(ascending=False)
print('Feature correlation with price (|r|):')
print(price_corr)

**Insight:** `year` and `engineSize` tend to have the strongest positive correlations with price (newer and bigger engine = more expensive), while `mileage` is negatively correlated (more miles = cheaper). `tax` and `mpg` show moderate relationships.

### 1.6 Additional EDA — Categorical Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price by transmission type
df.boxplot(column='price', by='transmission', ax=axes[0, 0])
axes[0, 0].set_title('Price by Transmission')
axes[0, 0].set_xlabel('Transmission')
axes[0, 0].set_ylabel('Price')

# Price by fuel type
df.boxplot(column='price', by='fuelType', ax=axes[0, 1])
axes[0, 1].set_title('Price by Fuel Type')
axes[0, 1].set_xlabel('Fuel Type')
axes[0, 1].set_ylabel('Price')

# Top 15 makes by count
df['Make'].value_counts().head(15).plot.bar(ax=axes[1, 0], color='steelblue', edgecolor='black')
axes[1, 0].set_title('Top 15 Car Makes by Count')
axes[1, 0].tick_params(axis='x', rotation=45)

# Average price by make (top 15)
df.groupby('Make')['price'].mean().sort_values(ascending=False).head(15).plot.bar(
    ax=axes[1, 1], color='salmon', edgecolor='black')
axes[1, 1].set_title('Top 15 Makes by Average Price')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.suptitle('')  # Remove pandas auto-title
plt.tight_layout()
plt.show()

**Insight:** Automatic cars tend to be more expensive than manual ones. Diesel and hybrid fuel types often fetch higher prices. Premium makes (e.g., Audi, BMW, Mercedes) dominate both volume and average price.

---
## 2. Data Preprocessing (Leak-Free Pipeline)

To prevent **data leakage**, we follow the golden rule: **split first, then fit on train only, transform both**.

**Pipeline:**
1. Drop the `model` column (too many unique values — high cardinality causes overfitting)
2. **Train/test split (80/20)**
3. Handle missing values (fit imputer on train only)
4. Detect and cap outliers (IQR from train only)
5. Encode categorical features with **One-Hot Encoding** (fit on train only)
6. Scale numerical features (fit on train only)

### 2.1 Drop High-Cardinality Column

In [ ]:
# 'model' has hundreds of unique values — encoding it would add too many columns
# and cause overfitting. We drop it.
print(f"Unique values in 'model': {df['model'].nunique()}")
df.drop(columns=['model'], inplace=True)
print(f'Shape after dropping model: {df.shape}')

### 2.2 Train/Test Split (Before Any Transformation)

We split **before** imputation, outlier capping, encoding, or scaling to prevent data leakage. All transformations are fit on the training set only.

In [ ]:
# Drop rows where price (target) is NaN — can't train on missing targets
df = df.dropna(subset=['price'])

# Separate features and target BEFORE any transformation
X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Target — min: {y_train.min():.0f}, max: {y_train.max():.0f}, mean: {y_train.mean():.0f}')
print(f'NaN in y_train: {y_train.isna().sum()}, NaN in y_test: {y_test.isna().sum()}')

### 2.3 Handle Missing Values (Fit on Train Only)

**Strategy:**
- **Numerical columns** → fill with the **median** (robust to outliers, better than mean for skewed data).
- **Categorical columns** → fill with the **mode** (most frequent value).

Both imputers are **fit on the training set only**, then used to transform both train and test.

In [ ]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

print(f'Numerical  ({len(num_cols)}): {num_cols}')
print(f'Categorical ({len(cat_cols)}): {cat_cols}')
print(f'\nMissing values BEFORE imputation (train):')
print(X_train.isnull().sum())

# Numerical imputer — fit on train only
num_imputer = SimpleImputer(strategy='median')
X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

# Categorical imputer — fit on train only
cat_imputer = SimpleImputer(strategy='most_frequent')
X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

print(f'\nMissing values AFTER imputation (train): {X_train.isnull().sum().sum()}')
print(f'Missing values AFTER imputation (test):  {X_test.isnull().sum().sum()}')

### 2.4 Outlier Detection and Capping (IQR Method — Train Statistics Only)

Extreme outliers distort model training (especially linear regression). We use the **IQR method** to cap values at [Q1 - 1.5×IQR, Q3 + 1.5×IQR].

**Critical:** IQR bounds are computed from the **training set only**, then applied to both train and test to prevent data leakage.

In [ ]:
def cap_outliers_iqr(train_df, test_df, column):
    """Cap outliers using IQR bounds computed from training data only."""
    Q1 = train_df[column].quantile(0.25)
    Q3 = train_df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_train = ((train_df[column] < lower) | (train_df[column] > upper)).sum()
    n_test  = ((test_df[column] < lower) | (test_df[column] > upper)).sum()
    train_df[column] = train_df[column].clip(lower, upper)
    test_df[column]  = test_df[column].clip(lower, upper)
    return n_train, n_test

# Also cap outliers in the target variable (y_train bounds applied to both)
def cap_target_outliers(y_train, y_test):
    Q1 = y_train.quantile(0.25)
    Q3 = y_train.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_train = ((y_train < lower) | (y_train > upper)).sum()
    n_test  = ((y_test < lower) | (y_test > upper)).sum()
    y_train = y_train.clip(lower, upper)
    y_test  = y_test.clip(lower, upper)
    print(f'price: {n_train} train / {n_test} test outliers capped  (bounds: {lower:.0f} – {upper:.0f})')
    return y_train, y_test

# Cap feature outliers
outlier_cols = ['mileage', 'tax', 'mpg', 'engineSize']
for c in outlier_cols:
    n_tr, n_te = cap_outliers_iqr(X_train, X_test, c)
    print(f'{c}: {n_tr} train / {n_te} test outliers capped')

# Cap target outliers
y_train, y_test = cap_target_outliers(y_train, y_test)

### 2.5 Encode Categorical Features (One-Hot Encoding)

We use **One-Hot Encoding** for `transmission`, `fuelType`, and `Make`.

**Why One-Hot instead of Label Encoding?**
- These are **nominal** features — there is no natural order among car makes, fuel types, or transmission types.
- Label Encoding assigns arbitrary numbers (e.g., Audi=0, BMW=1, Ford=2) which **Linear Regression and KNN would misinterpret as ordinal** (Ford > BMW > Audi).
- One-Hot creates separate binary columns per category, treating each independently.
- With only **7 makes**, **4 transmissions**, and **4 fuel types**, One-Hot adds only ~15 columns — very manageable.

The encoder is **fit on training data only** with `handle_unknown='ignore'` so any unseen categories in the test set produce all-zero rows instead of crashing.

In [ ]:
# One-Hot Encode categorical columns — fit on train only
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')

train_encoded = pd.DataFrame(
    ohe.fit_transform(X_train[cat_cols]),
    columns=ohe.get_feature_names_out(cat_cols),
    index=X_train.index
)
test_encoded = pd.DataFrame(
    ohe.transform(X_test[cat_cols]),
    columns=ohe.get_feature_names_out(cat_cols),
    index=X_test.index
)

# Drop original categorical columns and attach encoded ones
X_train = X_train.drop(columns=cat_cols).join(train_encoded)
X_test  = X_test.drop(columns=cat_cols).join(test_encoded)

print(f'Categories encoded: {cat_cols}')
print(f'New feature names: {list(ohe.get_feature_names_out(cat_cols))}')
print(f'Train shape: {X_train.shape}  |  Test shape: {X_test.shape}')
X_train.head()

### 2.6 Feature Scaling (StandardScaler — Fit on Train Only)

KNN relies on distances — features with larger ranges would dominate the distance calculation. Linear regression also benefits from standardised inputs for numerical stability.

We use **StandardScaler** (z-score: (x − μ) / σ) fitted on the **training set only** to prevent data leakage.

In [ ]:
# Save unscaled copies for sensitivity analysis later
X_train_unscaled = X_train.copy()
X_test_unscaled  = X_test.copy()

# Scale all features — fit on train only
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)

print(f'Scaled train shape: {X_train_scaled.shape}')
print(f'Scaled test shape:  {X_test_scaled.shape}')
print(f'\nPreprocessing complete — no data leakage!')
X_train_scaled.head()

---
## 3. Target Variables

We define two targets from the same preprocessed data:
- **Regression:** `y_train` / `y_test` (continuous price — already split)
- **Classification:** `y_train_cls` / `y_test_cls` (Cheap / Moderate / Expensive — derived from price using **training set** percentiles)

### 3A. Regression Target

`y_train` and `y_test` are already defined from the split in Section 2.2.

In [ ]:
print(f'Regression target (y_train):')
print(f'  Samples: {len(y_train)}')
print(f'  Min: {y_train.min():.0f}, Max: {y_train.max():.0f}, Mean: {y_train.mean():.0f}')
print(f'\nRegression target (y_test):')
print(f'  Samples: {len(y_test)}')
print(f'  Min: {y_test.min():.0f}, Max: {y_test.max():.0f}, Mean: {y_test.mean():.0f}')

### 3B. Classification Target

We create three price categories using the **33rd and 67th percentiles** computed from the **training set only** (to avoid leaking test distribution information).

This approach:
- Is **data-driven** (not arbitrary fixed numbers).
- Produces roughly **equal-sized classes** in the training set, which prevents class imbalance.
- Training percentiles are applied to both train and test — the test set class distribution may differ slightly, which is realistic.

In [ ]:
# Compute percentile thresholds from TRAINING data only
t1 = y_train.quantile(0.33)
t2 = y_train.quantile(0.67)
print(f'Thresholds (from training set): Cheap < {t1:.0f} | Moderate: {t1:.0f}–{t2:.0f} | Expensive > {t2:.0f}')

def categorise_price(price, t1, t2):
    if price <= t1:
        return 0  # Cheap
    elif price <= t2:
        return 1  # Moderate
    else:
        return 2  # Expensive

label_map = {0: 'Cheap', 1: 'Moderate', 2: 'Expensive'}

# Apply thresholds (computed from train) to both train and test
y_train_cls = y_train.apply(lambda p: categorise_price(p, t1, t2))
y_test_cls  = y_test.apply(lambda p: categorise_price(p, t1, t2))

print(f'\nTraining class distribution:')
print(y_train_cls.map(label_map).value_counts().sort_index())
print(f'\nTest class distribution:')
print(y_test_cls.map(label_map).value_counts().sort_index())

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, title in [(axes[0], y_train_cls, 'Train'), (axes[1], y_test_cls, 'Test')]:
    data.map(label_map).value_counts().sort_index().plot.bar(
        ax=ax, color=['#2ecc71', '#3498db', '#e74c3c'], edgecolor='black')
    ax.set_title(f'{title} — Price Category Distribution', fontsize=12)
    ax.set_xlabel('Category')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

# Visualise thresholds on training price distribution
plt.figure(figsize=(10, 5))
plt.hist(y_train, bins=50, edgecolor='black', color='steelblue', alpha=0.7)
plt.axvline(t1, color='red', linestyle='--', linewidth=2, label=f'33rd pctl = {t1:.0f}')
plt.axvline(t2, color='green', linestyle='--', linewidth=2, label=f'67th pctl = {t2:.0f}')
plt.title('Training Price Distribution with Classification Thresholds', fontsize=14)
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 4. Model 1: Linear Regression

We predict the **continuous price** using the preprocessed, scaled features from Section 2.

The data is already split, imputed, encoded, and scaled — we go straight to training.

In [ ]:
# Train Linear Regression on the preprocessed scaled data
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predict on test set
y_pred_r = lr_model.predict(X_test_scaled)
print('Linear Regression trained successfully.')

In [ ]:
# Evaluation metrics
mae  = mean_absolute_error(y_test, y_pred_r)
mse  = mean_squared_error(y_test, y_pred_r)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred_r)

print('=== Linear Regression Results ===')
print(f'MAE  : {mae:.2f}')
print(f'MSE  : {mse:.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'R2   : {r2:.4f}')

In [ ]:
# Feature importances (coefficients)
coef_df = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print('Feature coefficients (standardised):')
coef_df

### 4.1 Predicted vs Actual Plot

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_r, alpha=0.15, s=10, color='steelblue')

# Perfect prediction line
lims = [min(y_test.min(), y_pred_r.min()), max(y_test.max(), y_pred_r.max())]
plt.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')

plt.title('Linear Regression — Predicted vs Actual Price', fontsize=14)
plt.xlabel('Actual Price', fontsize=12)
plt.ylabel('Predicted Price', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Model 2: KNN Classification

We classify cars into **Cheap / Moderate / Expensive** using K-Nearest Neighbours.

We use **GridSearchCV** with **5-fold cross-validation** to tune:
- `n_neighbors`: 3, 5, 7, 9, 11, 13, 15
- `metric`: euclidean, manhattan

In [ ]:
# GridSearchCV to find optimal k and distance metric
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15],
    'metric': ['euclidean', 'manhattan']
}

knn = KNeighborsClassifier()
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train_scaled, y_train_cls)

print(f'Best Parameters: {grid_search.best_params_}')
print(f'Best CV Accuracy: {grid_search.best_score_:.4f}')

In [ ]:
# Plot GridSearchCV results
results = pd.DataFrame(grid_search.cv_results_)

plt.figure(figsize=(10, 6))
for metric in ['euclidean', 'manhattan']:
    subset = results[results['param_metric'] == metric]
    plt.plot(subset['param_n_neighbors'], subset['mean_test_score'],
             label=f'{metric}', marker='o')

best_k = grid_search.best_params_['n_neighbors']
plt.axvline(best_k, color='red', linestyle='--', alpha=0.7,
            label=f'Best k={best_k}')
plt.title('KNN — CV Accuracy vs Number of Neighbours', fontsize=14)
plt.xlabel('k (Number of Neighbours)', fontsize=12)
plt.ylabel('Mean CV Accuracy', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate best model on test set
best_knn = grid_search.best_estimator_
y_pred_c = best_knn.predict(X_test_scaled)

acc  = accuracy_score(y_test_cls, y_pred_c)
prec = precision_score(y_test_cls, y_pred_c, average='weighted')
rec  = recall_score(y_test_cls, y_pred_c, average='weighted')
f1   = f1_score(y_test_cls, y_pred_c, average='weighted')

print('=== KNN Classification Results ===')
print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1 Score  : {f1:.4f}')
print()
print(classification_report(y_test_cls, y_pred_c,
                            target_names=['Cheap', 'Moderate', 'Expensive']))

### 5.1 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test_cls, y_pred_c)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Cheap', 'Moderate', 'Expensive'],
            yticklabels=['Cheap', 'Moderate', 'Expensive'])
plt.title('KNN — Confusion Matrix', fontsize=14)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.show()

**Interpretation:** The diagonal shows correct predictions. Most confusion typically occurs between adjacent categories (Cheap vs Moderate, Moderate vs Expensive), which makes sense because cars near the threshold boundary are inherently hard to classify.

---
## 6. Analysis & Discussion

### 6.1 Comparing Regression vs Classification

| Aspect | Linear Regression | KNN Classification |
|--------|-------------------|---------------------|
| **Task** | Predict exact price | Predict price category |
| **Key metric** | R2 (how much variance explained) | Accuracy / F1 |
| **Interpretability** | Coefficients show feature importance directly | No explicit feature importance |

**Is classification easier?** Generally yes — the classifier only needs to place a car into one of three broad buckets rather than pinpointing an exact price. A car that the regressor predicts as 15 500 when the truth is 16 000 (500 error) would still be classified correctly as "Moderate" by the classifier.

**Does categorising price lose information?** Yes — by converting a continuous target into 3 bins, we discard the within-bin variation. Two cars priced at 8 000 and 12 000 might both be labelled "Cheap," yet they are quite different for a buyer. Regression preserves this granularity.

### 6.2 Sensitivity Analysis

We test how the models respond to three changes:
1. **Remove the most correlated feature** and retrain both models.
2. **Run KNN without scaling** and compare performance.
3. **Change the classification thresholds** and observe the impact.

#### SA-1: Remove Most Correlated Feature

In [ ]:
# Identify the feature most correlated with price (using training data)
# We need to combine X_train (before scaling) with y_train for correlation
train_with_target = X_train_unscaled.copy()
train_with_target['price'] = y_train.values
corr_with_price = train_with_target.corr()['price'].drop('price').abs().sort_values(ascending=False)
top_feature = corr_with_price.idxmax()
print(f'Most correlated feature with price: "{top_feature}" (|r| = {corr_with_price[top_feature]:.4f})')

# Create reduced feature sets (drop top feature, re-scale)
X_train_reduced = X_train_scaled.drop(columns=[top_feature])
X_test_reduced  = X_test_scaled.drop(columns=[top_feature])

# --- Regression without top feature ---
lr_reduced = LinearRegression().fit(X_train_reduced, y_train)
y_pred_red = lr_reduced.predict(X_test_reduced)
r2_reduced   = r2_score(y_test, y_pred_red)
rmse_reduced = np.sqrt(mean_squared_error(y_test, y_pred_red))

print(f'\nLinear Regression — ALL features:  R2={r2:.4f}, RMSE={rmse:.2f}')
print(f'Linear Regression — WITHOUT "{top_feature}": R2={r2_reduced:.4f}, RMSE={rmse_reduced:.2f}')
print(f'R2 dropped by {(r2 - r2_reduced):.4f}')

# --- KNN without top feature ---
knn_reduced = KNeighborsClassifier(**grid_search.best_params_).fit(X_train_reduced, y_train_cls)
y_pred_c_red = knn_reduced.predict(X_test_reduced)
acc_reduced = accuracy_score(y_test_cls, y_pred_c_red)

print(f'\nKNN — ALL features:  Accuracy={acc:.4f}')
print(f'KNN — WITHOUT "{top_feature}": Accuracy={acc_reduced:.4f}')
print(f'Accuracy dropped by {(acc - acc_reduced):.4f}')

**Takeaway:** Removing the most predictive feature hurts both models, confirming that it carries important pricing information.

#### SA-2: KNN Without Scaling

In [ ]:
# KNN on UNSCALED features (using the unscaled copies saved before scaling)
knn_noscale = KNeighborsClassifier(**grid_search.best_params_)
knn_noscale.fit(X_train_unscaled, y_train_cls)
y_pred_noscale = knn_noscale.predict(X_test_unscaled)

acc_noscale = accuracy_score(y_test_cls, y_pred_noscale)
f1_noscale  = f1_score(y_test_cls, y_pred_noscale, average='weighted')

print('=== KNN: Scaled vs Unscaled ===')
print(f'WITH scaling    — Accuracy: {acc:.4f}, F1: {f1:.4f}')
print(f'WITHOUT scaling — Accuracy: {acc_noscale:.4f}, F1: {f1_noscale:.4f}')
print(f'Accuracy difference: {(acc - acc_noscale):.4f}')

**Takeaway:** Without scaling, features with large ranges (e.g., `mileage` in thousands) dominate the distance calculation, drowning out smaller-range features (e.g., `engineSize`). This confirms that **scaling is essential for KNN**.

#### SA-3: Change Classification Thresholds

In [ ]:
# Test alternative thresholds: 25th / 75th percentile (from training data only)
t1_alt = y_train.quantile(0.25)
t2_alt = y_train.quantile(0.75)

y_train_cls_alt = y_train.apply(lambda p: categorise_price(p, t1_alt, t2_alt))
y_test_cls_alt  = y_test.apply(lambda p: categorise_price(p, t1_alt, t2_alt))

print(f'Original thresholds (33/67): {t1:.0f} / {t2:.0f}')
print(f'Alternative thresholds (25/75): {t1_alt:.0f} / {t2_alt:.0f}')
print(f'\nAlternative training class distribution:')
print(y_train_cls_alt.map(label_map).value_counts().sort_index())

# Retrain KNN with alternative thresholds
knn_alt = KNeighborsClassifier(**grid_search.best_params_)
knn_alt.fit(X_train_scaled, y_train_cls_alt)
y_pred_alt = knn_alt.predict(X_test_scaled)

acc_alt = accuracy_score(y_test_cls_alt, y_pred_alt)
f1_alt  = f1_score(y_test_cls_alt, y_pred_alt, average='weighted')

print(f'\nKNN — Original thresholds (33/67):  Accuracy={acc:.4f}, F1={f1:.4f}')
print(f'KNN — Alternative thresholds (25/75): Accuracy={acc_alt:.4f}, F1={f1_alt:.4f}')

**Takeaway:** With 25/75 thresholds, the Moderate class expands (50% of data) while Cheap and Expensive shrink (25% each). The overall accuracy may go up because the bigger middle class is easier to predict, but per-class performance becomes less balanced.

---
## 7. Visualisations (Summary)

Mandatory plots already shown above:
1. Price histogram (Section 1.4)
2. Correlation heatmap (Section 1.5)
3. Predicted vs Actual scatter (Section 4.1)
4. Confusion matrix heatmap (Section 5.1)

Below are **two additional meaningful plots**.

### Extra Plot 1: Residual Distribution (Linear Regression)

Residuals (actual - predicted) should be approximately normally distributed and centred around zero for a good regression model.

In [ ]:
residuals = y_test.values - y_pred_r

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of residuals
axes[0].hist(residuals, bins=50, edgecolor='black', color='steelblue', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_title('Residual Distribution', fontsize=14)
axes[0].set_xlabel('Residual (Actual - Predicted)')
axes[0].set_ylabel('Frequency')

# Residuals vs Predicted
axes[1].scatter(y_pred_r, residuals, alpha=0.1, s=10, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_title('Residuals vs Predicted Values', fontsize=14)
axes[1].set_xlabel('Predicted Price')
axes[1].set_ylabel('Residual')

plt.tight_layout()
plt.show()

print(f'Mean residual: {residuals.mean():.2f} (should be ~0)')
print(f'Std residual:  {residuals.std():.2f}')

**Insight:** If residuals fan out at higher predicted values, it indicates **heteroscedasticity** — the model's error grows with price. This is a known limitation of linear regression on skewed price data.

### Extra Plot 2: Feature Importance Comparison

We compare the absolute standardised regression coefficients to understand which features drive the price prediction most.

In [ ]:
# Bar chart of absolute coefficients
coef_df_sorted = coef_df.copy()
coef_df_sorted['AbsCoefficient'] = coef_df_sorted['Coefficient'].abs()
coef_df_sorted = coef_df_sorted.sort_values('AbsCoefficient', ascending=True)

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coef_df_sorted['Coefficient']]
plt.barh(coef_df_sorted['Feature'], coef_df_sorted['Coefficient'], color=colors, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Linear Regression — Feature Coefficients (Standardised)', fontsize=14)
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

**Insight:** Green bars (positive coefficients) push price up; red bars (negative) push it down. This gives an intuitive, interpretable summary of what the linear model learned about car pricing.

---
## Summary

| Metric | Linear Regression | KNN Classification |
|--------|-------------------|---------------------|
| **Primary metric** | R2 | Accuracy |
| **Scaling needed?** | Helpful | Essential |
| **Hyperparameter tuning** | None (closed-form) | GridSearchCV over k and distance |
| **Interpretability** | High (coefficients) | Low (instance-based) |

**Key findings:**
- `year` and `engineSize` are among the strongest price predictors.
- KNN classification benefits enormously from feature scaling.
- Converting price to categories makes the task "easier" but loses within-bin granularity.
- Removing the top correlated feature visibly degrades both models.

**Methodology (leak-free pipeline):**
- Train/test split is performed **before** any transformation.
- All imputers, encoders, scalers, and outlier bounds are **fit on training data only**.
- **One-Hot Encoding** is used for nominal features (Make, transmission, fuelType) — no false ordinal relationships.
- Classification thresholds are derived from **training set** percentiles only.

---
## Summary

| Metric | Linear Regression | KNN Classification |
|--------|-------------------|--------------------|
| **Primary metric** | R² | Accuracy |
| **Scaling needed?** | Helpful | Essential |
| **Hyperparameter tuning** | None (closed-form) | GridSearchCV over k and distance |
| **Interpretability** | High (coefficients) | Low (instance-based) |

**Key findings:**
- `year` and `engineSize` are among the strongest price predictors.
- KNN classification benefits enormously from feature scaling.
- Converting price to categories makes the task "easier" but loses within-bin granularity.
- Removing the top correlated feature visibly degrades both models.